# Part 1 · Notebook 04 — Futures and options calculator

**Sessions:** S10 (futures), S11 (options), S12–S13 (strategies), S15 (Excel models) · **Use it to check your Excel workbook.**

Every number here should match your Excel sheets. If they differ, find out why (usually units: % vs decimal, days vs years, per-day theta).

In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))          # p1lib.py lives next to this notebook
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p1lib as p

p.use_course_style()
OUT_DIR = Path("outputs"); OUT_DIR.mkdir(exist_ok=True)   # CSV exports for Excel go here
pd.set_option("display.float_format", "{:,.4f}".format)
print("Offline fixtures:" if os.environ.get("P1_FIXTURES") else "Live data from FRED", os.environ.get("P1_FIXTURES", ""))

## 1. Futures fair value and basis

✏️ **Change me:** index level, interest rate, dividend yield, days to expiry, market price.

In [ ]:
SPOT, RATE, DIV_YIELD, DAYS = 6000.0, 0.043, 0.013, 80       # ✏️ Change me
MARKET_FUTURES = 6041.0                                       # ✏️ Change me
fv = p.futures_fair_value(SPOT, RATE, DIV_YIELD, DAYS)
print(f"Fair value: {fv:,.2f}   basis (F - S): {fv - SPOT:,.2f}   market - fair: {MARKET_FUTURES - fv:+.2f} points")

## 2. Daily variation margin

A long position is marked to market every day. A margin call brings the account back to initial margin.

In [ ]:
SETTLES = [6000, 5975, 5940, 5890, 5920, 5960, 5905, 5850, 5880, 5930]   # ✏️ Change me
ledger = p.margin_ledger(SETTLES, entry_price=6000, contracts=1, multiplier=50,        # ES: $50 per point
                         initial_margin=25_000, maintenance_margin=22_000)              # ✏️ check your broker's margins
fig, ax = plt.subplots()
ledger["balance"].plot(ax=ax, marker="o", title="Margin account balance ($), 1 ES contract")
ax.axhline(22_000, color="#8a8984", lw=1, ls="--"); ax.text(1, 22_150, "maintenance", color="#52514e", fontsize=9)
ax.set_xlabel("day"); plt.show()
ledger

## 3. Black–Scholes–Merton price and Greeks (check your Excel sheet)

In [ ]:
S, K, T, R, Q, VOL = 100.0, 105.0, 0.5, 0.04, 0.01, 0.25       # ✏️ Change me
res = p.bsm(S, K, T, R, Q, VOL)
pd.Series(res).round(6)

In [ ]:
# Expected with the default inputs (same as the lesson's verified Excel sheet):
expected = {"call": 5.5482, "put": 8.9678, "delta_call": 0.4568, "gamma": 0.02234,
            "vega_per_point": 0.2792, "theta_call_per_day": -0.0223}
if (S, K, T, R, Q, VOL) == (100.0, 105.0, 0.5, 0.04, 0.01, 0.25):
    for k, v in expected.items():
        print(f"{k:<20}{res[k]:>10.4f}  expected {v:>9.4f}  {'OK' if abs(res[k] - v) < 1e-4 else 'CHECK'}")

## 4. Implied volatility (Excel: Goal Seek)

✏️ Enter an option's market price and solve for the volatility that reproduces it.

In [ ]:
MARKET_PRICE, KIND = 6.10, "call"       # ✏️ Change me
iv = p.implied_vol(MARKET_PRICE, S, K, T, R, Q, KIND)
print(f"Implied volatility: {iv:.2%}")

## 5. How the price responds to each input

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
spots = np.linspace(70, 140, 200)
for years, color in [(0.5, p.PALETTE[0]), (0.1, p.PALETTE[1]), (0.01, p.PALETTE[2])]:
    axes[0].plot(spots, [p.bsm(x, K, years, R, Q, VOL)["call"] for x in spots], color=color, label=f"T = {years} y")
axes[0].set_title("Call value vs spot"); axes[0].legend()
vols = np.linspace(0.05, 0.8, 100)
axes[1].plot(vols * 100, [p.bsm(S, K, T, R, Q, v)["call"] for v in vols]); axes[1].set_title("Call value vs volatility (%)")
days = np.arange(180, 0, -1)
axes[2].plot(days, [p.bsm(S, K, d / 365, R, Q, VOL)["call"] for d in days]); axes[2].invert_xaxis()
axes[2].set_title("Call value as expiry approaches (days left)")
plt.tight_layout(); plt.show()

## 6. Payoff builder

✏️ Define strategies as lists of legs. `premium` = price paid per unit (options) or entry price (stock). Premiums below come from the BSM calculator above.

In [ ]:
price = lambda k, kind: p.bsm(S, k, T, R, Q, VOL)[kind]
STRATEGIES = {                                                     # ✏️ Change me / add your own
    "Covered call": [{"type": "stock", "qty": 1, "premium": S},
                     {"type": "call", "qty": -1, "strike": 110, "premium": price(110, "call")}],
    "Collar": [{"type": "stock", "qty": 1, "premium": S},
               {"type": "put", "qty": 1, "strike": 90, "premium": price(90, "put")},
               {"type": "call", "qty": -1, "strike": 110, "premium": price(110, "call")}],
    "Bull call spread": [{"type": "call", "qty": 1, "strike": 105, "premium": price(105, "call")},
                         {"type": "call", "qty": -1, "strike": 115, "premium": price(115, "call")}],
    "Long straddle": [{"type": "call", "qty": 1, "strike": 100, "premium": price(100, "call")},
                      {"type": "put", "qty": 1, "strike": 100, "premium": price(100, "put")}],
    "Iron condor": [{"type": "put", "qty": 1, "strike": 85, "premium": price(85, "put")},
                    {"type": "put", "qty": -1, "strike": 90, "premium": price(90, "put")},
                    {"type": "call", "qty": -1, "strike": 110, "premium": price(110, "call")},
                    {"type": "call", "qty": 1, "strike": 115, "premium": price(115, "call")}],
    "Synthetic long": [{"type": "call", "qty": 1, "strike": 100, "premium": price(100, "call")},
                       {"type": "put", "qty": -1, "strike": 100, "premium": price(100, "put")}],
}
grid = np.linspace(60, 140, 801)
fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharex=True)
rows = []
for ax, (name, legs) in zip(axes.flat, STRATEGIES.items()):
    pnl = p.payoff_at_expiry(legs, grid)
    ax.plot(grid, pnl, color=p.PALETTE[0]); ax.axhline(0, color="#52514e", lw=1); ax.set_title(name)
    be = p.breakevens(grid, pnl)
    for b in be:
        ax.axvline(b, color="#8a8984", lw=1, ls=":")
    rows.append({"strategy": name, "max_profit (grid)": pnl.max(), "max_loss (grid)": pnl.min(),
                 "breakevens": ", ".join(f"{b:.2f}" for b in be)})
fig.suptitle("P&L at expiry per unit (vertical dotted lines = breakevens)", fontweight="bold")
plt.tight_layout(); plt.show()
pd.DataFrame(rows).set_index("strategy")

> Max profit/loss are measured on the 60–140 grid: strategies with unlimited upside or downside show the grid edge, not infinity.

## 7. OCC option symbols

In [ ]:
from datetime import date
print(p.occ_symbol("SPY", date(2026, 12, 18), "C", 600))    # ✏️ try your own

## 8. Bonus (S5 homework): leveraged ETF decay

A 2× daily leveraged fund in a choppy market that ends flat.

In [ ]:
rng = np.random.default_rng(3)
daily = rng.normal(0, 0.02, 60); daily -= daily.mean()             # zero average daily return
index = np.cumprod(1 + daily); lev2 = np.cumprod(1 + 2 * daily)
out = pd.DataFrame({"Index": index, "2x daily ETF": lev2})
ax = out.plot(title="Index vs 2× daily leveraged ETF (60 choppy days)"); ax.set_xlabel("day"); plt.show()
print(f"Index return: {index[-1] - 1:+.2%}   2x of that: {2 * (index[-1] - 1):+.2%}   2x ETF actual: {lev2[-1] - 1:+.2%}")
p.log_research({"notebook": "04_derivatives_calculator", "note": "checked Excel models"});

## Questions
1. Your Excel BSM sheet and this notebook disagree in the 4th decimal. What are the three most likely causes?
2. Which strategies above have defined risk? Which one is equivalent to owning the stock, and why (put-call parity)?
3. Why does the leveraged ETF lose value even though the index ends roughly flat?